## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions


In [3]:
import os 
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROK_API_KEY")

groq_api_key

In [4]:
from langchain_groq import ChatGroq

model=ChatGroq(model="gemma2-9b-it",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000195F4794E10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000195F4883790>, model_name='gemma2-9b-it', model_kwargs={})

In [5]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is yash I'm a generative AI learner")])

AIMessage(content="Hello Yash! It's great to meet you.  \n\nBeing a generative AI learner is fascinating! What aspects of generative AI are you most interested in learning about? \n\nI can help you explore various topics, such as:\n\n* **Different types of generative models:**  (e.g., GANs, VAEs, Transformers)\n* **Applications of generative AI:** (e.g., text generation, image synthesis, music composition)\n* **Ethical considerations:** (e.g., bias, misuse, copyright)\n* **The technical details:** (e.g., how models are trained, architectures)\n\nJust let me know what sparks your curiosity! 😄  \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 23, 'total_tokens': 167, 'completion_time': 0.261818182, 'prompt_time': 0.0013218, 'queue_time': 0.25044093, 'total_time': 0.263139982}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--0e435

In [6]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, My name is Yash and i'm a generative AI engineer."),
        AIMessage(content="Hello Yash\n\nIt's great to meet you! I'm glad you're interested in generative AI. It's a fascinating field with a lot of potential. \n\nWhat specifically are you learning about generative AI? Are there any particular areas that interest you the most? \n\nI'm happy to answer any questions you have or discuss different aspects of generative AI. "),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content="You told me your name is Yash, and you are a generative AI engineer!  \n\nIs there anything else you'd like to tell me about yourself or your work? \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 124, 'total_tokens': 163, 'completion_time': 0.070909091, 'prompt_time': 0.003288069, 'queue_time': 0.25023459, 'total_time': 0.07419716}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--c04bafec-6416-47f8-aef8-14028e9cb64f-0', usage_metadata={'input_tokens': 124, 'output_tokens': 39, 'total_tokens': 163})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [8]:
config={"configurable":{"session_id":"chat1"}}

In [9]:
with_message_history.invoke(
    [HumanMessage(content="Hi, what is my name")],
    config=config
)

AIMessage(content="As an AI, I have no memory of past conversations and do not know your name. If you'd like to tell me, I'd be happy to use it! 😊  \n\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 15, 'total_tokens': 57, 'completion_time': 0.076363636, 'prompt_time': 0.001251149, 'queue_time': 0.25147067, 'total_time': 0.077614785}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--76277379-b3df-4fba-bde6-9d4feae83c07-0', usage_metadata={'input_tokens': 15, 'output_tokens': 42, 'total_tokens': 57})

In [10]:
# change the config -> session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi, what is my name")],
    config=config1
)
response.content

"As an AI, I have no memory of past conversations and do not know your name. If you'd like to tell me your name, I'd be happy to use it! 😊\n"

In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, my name is Yash")],
    config=config
)
response.content

"Hi Yash, it's nice to meet you!  \n\nWhat can I do for you today?\n"

In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, what is my name")],
    config=config
)
response.content

"As an AI, I don't have memory of past conversations.  \n\nEven though you told me your name was Yash earlier, I don't remember it now. If you'd like to tell me again, I'd be happy to use it! 😊  \n\n"

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="message")
    ]
)

chain=prompt|model

In [15]:
chain.invoke({"message":[HumanMessage(content="Hi my name is yash")]})

AIMessage(content="Hello Yash, it's nice to meet you! \n\nAsk me anything, I'm here to help. 😊  \n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 31, 'total_tokens': 60, 'completion_time': 0.052727273, 'prompt_time': 0.001474472, 'queue_time': 0.250396127, 'total_time': 0.054201745}, 'model_name': 'gemma2-9b-it', 'system_fingerprint': 'fp_10c08bf97d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--30f45f32-281d-43b3-b137-32dabc6db70f-0', usage_metadata={'input_tokens': 31, 'output_tokens': 29, 'total_tokens': 60})

In [16]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)


In [18]:
config={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="What is my name")],
    config=config   
)
response.content

'Your name is Yash, as you told me! 😊  \n\n'

In [ ]:
### add more complexity

prompt=ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all question to the best of your ability in {language}"
        ),
        MessagesPlaceholder(variable_name="message")
    ]
)
chain=prompt|model

In [24]:
response=chain.invoke({"message":[HumanMessage(content="Hi my name is yash")],"language":"spanish"})
response.content

'¡Hola Yash! Encantado de conocerte. ¿Cómo te puedo ayudar? \n'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages